# 📊 Dashboard de KPIs de Negocio con Machine Learning
### Dataset: Amazon Sale Report

Este notebook calcula 4 KPIs de negocio, cada uno con su propio modelo de Machine Learning,
siguiendo exactamente la tabla de referencia:

| KPI | Machine Learning | Variables principales |
|---|---|---|
| Ingresos por Categoría de Producto | Regresión Lineal | Category, Qty, Status, Amount |
| Tasa de Cancelación de Pedidos | Árbol de Decisión | Category, Qty, Amount, Fulfilment, ship-state, Status |
| Ticket Promedio por Pedido | K-Means (Clustering) | Order ID, Qty, Amount |
| Participación de Ingresos por Categoría | Clustering Jerárquico | Category, Amount, Qty, Status |

Ejecuta las celdas **en orden**, de arriba hacia abajo.

## 1️⃣ Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.cluster import KMeans, AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, classification_report,
    mean_squared_error, r2_score, confusion_matrix
)

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

## 2️⃣ Cargar el dataset

**No uses el botón de subida interactivo** (`files.upload()`) — en muchos navegadores
se queda "cargando" para siempre. En su lugar:

1. Abre el panel de la izquierda, ícono de **carpeta** 📁 ("Archivos").
2. Arrastra tu archivo (Excel o CSV) directamente ahí adentro (o usa el ícono de
   subir archivo que aparece arriba de ese panel).
3. Espera a que termine de subir (barra de progreso azul).
4. Haz clic derecho sobre el archivo ya subido → **"Copiar ruta"**.
5. Pega esa ruta reemplazando el valor de `NOMBRE_ARCHIVO` en la celda de abajo.

In [ ]:
NOMBRE_ARCHIVO = "/content/Amazon Sale Report.csv"  # <-- reemplaza por la ruta que copiaste

import zipfile

with open(NOMBRE_ARCHIVO, "rb") as f:
    firma = f.read(2)

# Nota: algunos archivos ".csv" descargados de ciertas fuentes (o exportados
# desde Excel/Drive) en realidad son un ZIP disfrazado con extensión .csv.
# Este bloque lo detecta automáticamente (firma "PK") y extrae el CSV real
# para evitar el error "UnicodeDecodeError" / "invalid start byte".
if firma == b"PK":
    with zipfile.ZipFile(NOMBRE_ARCHIVO) as zf:
        nombre_interno = zf.namelist()[0]
        with zf.open(nombre_interno) as f:
            df_raw = pd.read_csv(f, low_memory=False)
elif NOMBRE_ARCHIVO.lower().endswith((".xlsx", ".xls")):
    df_raw = pd.read_excel(NOMBRE_ARCHIVO)
else:
    df_raw = pd.read_csv(NOMBRE_ARCHIVO, low_memory=False)

print(f"✅ Archivo cargado: {NOMBRE_ARCHIVO}")
print("Dimensiones:", df_raw.shape)
df_raw.head()

## 3️⃣ Exploración inicial (diagnóstico de columnas)

Revisamos nulos, valores únicos y tipo de dato de cada columna para decidir qué se limpia.

In [ ]:
diagnostico = pd.DataFrame({
    "dtype": df_raw.dtypes,
    "n_nulos": df_raw.isna().sum(),
    "%_nulos": (df_raw.isna().sum() / len(df_raw) * 100).round(2),
    "n_valores_unicos": df_raw.nunique(),
})
diagnostico

## 4️⃣ Limpieza de datos

Se eliminan columnas que **no aportan contenido útil** para el análisis:

- **`New`** y **`PendingS`**: 100% de valores nulos (vacías por completo).
- **`currency`**: un único valor (`INR`) en todo el dataset → sin variabilidad.
- **`ship-country`**: un único valor (`IN`) en todo el dataset → sin variabilidad.
- **`fulfilled-by`**: ~70% nula y el único valor presente (`Easy Ship`) no discrimina nada.
- **`index`**: columna redundante, duplica el índice propio del DataFrame.

*(Ajusta esta lista si tu archivo tiene nombres de columnas distintos)*

In [ ]:
COLUMNAS_SIN_CONTENIDO = ["New", "PendingS", "currency", "ship-country", "fulfilled-by", "index"]
columnas_a_eliminar = [c for c in COLUMNAS_SIN_CONTENIDO if c in df_raw.columns]

df = df_raw.drop(columns=columnas_a_eliminar)
df = df.drop_duplicates()  # elimina filas 100% duplicadas

# Normalización de tipos y nulos en las columnas que sí se usan
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')
df['Qty'] = pd.to_numeric(df['Qty'], errors='coerce')
df['Category'] = df['Category'].fillna('Desconocido')
df['Status'] = df['Status'].fillna('Desconocido')
df['Fulfilment'] = df['Fulfilment'].fillna('Desconocido')
df['ship-state'] = df['ship-state'].fillna('Desconocido')

# Variable derivada: pedido cancelado (1) o no (0), usada en el KPI 2
df['Es_Cancelado'] = df['Status'].apply(lambda x: 1 if 'Cancelled' in str(x) else 0)

print(f"Columnas eliminadas: {columnas_a_eliminar}")
print(f"Shape original: {df_raw.shape}  ->  Shape tras limpieza: {df.shape}")
df.head()

---
## 📌 KPI 1: Ingresos por Categoría de Producto
### 🤖 Modelo: Regresión Lineal
**Variable objetivo:** `Amount` &nbsp;&nbsp; **Predictoras:** `Category`, `Qty`, `Status`

In [ ]:
df_kpi1 = df.dropna(subset=['Amount', 'Qty']).copy()

le_cat = LabelEncoder()
le_status = LabelEncoder()
df_kpi1['Category_code'] = le_cat.fit_transform(df_kpi1['Category'])
df_kpi1['Status_code'] = le_status.fit_transform(df_kpi1['Status'])

X = df_kpi1[['Category_code', 'Qty', 'Status_code']]
y = df_kpi1['Amount']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo_reg = LinearRegression()
modelo_reg.fit(X_train, y_train)
y_pred = modelo_reg.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("===== KPI 1: Ingresos por Categoría (Regresión Lineal) =====")
print(f"R² Score (explicación de ventas): {r2:.4f}")
print(f"RMSE (error medio en ingresos):   ${rmse:.2f}")

In [ ]:
# Ingresos totales y promedio por categoría (referencia directa, no es predicción)
resumen_categoria = df_kpi1.groupby('Category')['Amount'].agg(
    Ingresos_Totales='sum', Ingreso_Promedio='mean', N_Pedidos='count'
).reset_index().sort_values('Ingresos_Totales', ascending=False)

display(resumen_categoria)

plt.figure(figsize=(10, 4))
sns.barplot(data=resumen_categoria, x='Category', y='Ingresos_Totales', hue='Category',
            palette='Blues_d', legend=False)
plt.xticks(rotation=45)
plt.ylabel("Ingresos Totales ($)")
plt.title("Ingresos Totales por Categoría")
plt.tight_layout()
plt.show()

---
## 📌 KPI 2: Tasa de Cancelación de Pedidos
### 🤖 Modelo: Árbol de Decisión (Clasificación)
**Variable objetivo:** `Es_Cancelado` (derivada de `Status`) &nbsp;&nbsp;
**Predictoras:** `Category`, `Fulfilment`, `ship-state` *(se excluyen `Qty`/`Amount` porque
quedan en 0/nulo DESPUÉS de una cancelación: usarlas provoca fuga de datos)*

In [ ]:
df_kpi2 = df.copy()

for col in ['Category', 'Fulfilment', 'ship-state']:
    le = LabelEncoder()
    df_kpi2[col + '_code'] = le.fit_transform(df_kpi2[col].astype(str))

features = ['Category_code', 'Fulfilment_code', 'ship-state_code']
X = df_kpi2[features]
y = df_kpi2['Es_Cancelado']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

arbol = DecisionTreeClassifier(max_depth=3, random_state=42, class_weight='balanced')
arbol.fit(X_train, y_train)
y_pred = arbol.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print("===== KPI 2: Tasa de Cancelación (Árbol de Decisión) =====")
print(f"Precisión del modelo (Accuracy): {acc*100:.2f}%\n")
print(classification_report(y_test, y_pred, target_names=['No Cancelado', 'Cancelado']))

In [ ]:
importancias = pd.DataFrame({
    'Variable': features,
    'Importancia': arbol.feature_importances_
}).sort_values('Importancia', ascending=False)
display(importancias)

plt.figure(figsize=(14, 6))
plot_tree(arbol, feature_names=features, class_names=['No Cancelado', 'Cancelado'],
          filled=True, fontsize=8)
plt.title("Árbol de Decisión: Predicción de Cancelación")
plt.show()

---
## 📌 KPI 3: Ticket Promedio por Pedido
### 🤖 Modelo: K-Means (Clustering)
**Variables:** `Qty`, `Amount` agregados por `Order ID`

In [ ]:
df_kpi3 = df.dropna(subset=['Amount']).copy()

# Un pedido (Order ID) puede tener varias líneas de producto: se agrega a
# nivel de pedido antes de aplicar el clustering.
pedidos = df_kpi3.groupby('Order ID').agg(
    Amount=('Amount', 'sum'),
    Qty=('Qty', 'sum'),
    Category=('Category', lambda x: x.mode()[0] if not x.mode().empty else 'Desconocido')
).reset_index()

N_CLUSTERS = 3  # Bajo, Medio, Alto valor

scaler = StandardScaler()
escaladas = scaler.fit_transform(pedidos[['Qty', 'Amount']])

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
pedidos['Cluster'] = kmeans.fit_predict(escaladas)

resumen_clusters = pedidos.groupby('Cluster').agg(
    Ticket_Promedio=('Amount', 'mean'),
    Cantidad_Promedio=('Qty', 'mean'),
    Total_Pedidos=('Order ID', 'count')
).reset_index().sort_values('Ticket_Promedio')

etiquetas = ['Bajo valor', 'Valor medio', 'Alto valor', 'Muy alto valor', 'Premium']
resumen_clusters['Etiqueta'] = etiquetas[:len(resumen_clusters)]

print("===== KPI 3: Ticket Promedio por Pedido (K-Means) =====")
display(resumen_clusters)

In [ ]:
muestra = pedidos.sample(min(3000, len(pedidos)), random_state=42)
plt.figure(figsize=(9, 5))
sns.scatterplot(data=muestra, x='Qty', y='Amount', hue='Cluster', palette='viridis', alpha=0.6)
plt.title("Segmentación del Ticket de Compra por Pedido (Amount vs Qty)")
plt.show()

---
## 📌 KPI 4: Participación de Ingresos por Categoría
### 🤖 Modelo: Clustering Jerárquico
**Variables:** `Category`, `Amount`, `Qty`, `Status`

In [ ]:
resumen_cat = df.groupby('Category').agg(
    Total_Ingresos=('Amount', 'sum'),
    Ticket_Promedio=('Amount', 'mean'),
    Total_Unidades=('Qty', 'sum'),
    Total_Pedidos=('Order ID', 'nunique'),
    Tasa_Cancelacion=('Es_Cancelado', 'mean')
).reset_index()

resumen_cat['Participacion_%'] = (
    resumen_cat['Total_Ingresos'] / resumen_cat['Total_Ingresos'].sum() * 100
).round(2)

features_cat = resumen_cat[['Total_Ingresos', 'Ticket_Promedio', 'Total_Unidades', 'Total_Pedidos']]
scaler = StandardScaler()
escaladas_cat = scaler.fit_transform(features_cat)

linked = linkage(escaladas_cat, method='ward')

plt.figure(figsize=(10, 5))
dendrogram(linked, labels=resumen_cat['Category'].values, orientation='top',
           distance_sort='descending', show_leaf_counts=True)
plt.title("Dendrograma: Categorías de Alta, Media y Baja Contribución")
plt.ylabel("Distancia Euclídea")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
N_NIVELES = 3
agglo = AgglomerativeClustering(n_clusters=N_NIVELES)
resumen_cat['Cluster'] = agglo.fit_predict(escaladas_cat)

orden = resumen_cat.groupby('Cluster')['Total_Ingresos'].mean().sort_values().index.tolist()
niveles = ['Baja contribución', 'Media contribución', 'Alta contribución', 'Muy alta contribución']
mapa_niveles = {cl: niveles[i] for i, cl in enumerate(orden)}
resumen_cat['Nivel'] = resumen_cat['Cluster'].map(mapa_niveles)

print("===== KPI 4: Participación de Ingresos por Categoría (Clustering Jerárquico) =====")
display(
    resumen_cat.sort_values('Total_Ingresos', ascending=False)[
        ['Category', 'Total_Ingresos', 'Participacion_%', 'Tasa_Cancelacion', 'Nivel']
    ]
)

---
## 📝 Resumen: columnas eliminadas y por qué

- **`New`** → 100% vacía, ningún dato.
- **`PendingS`** → 100% vacía, ningún dato.
- **`currency`** → siempre el mismo valor (`INR`), no aporta.
- **`ship-country`** → siempre el mismo valor (`IN`), no aporta.
- **`fulfilled-by`** → ~70% nula y el único valor presente no discrimina.
- **`index`** → duplica el índice del propio DataFrame, es redundante.

Todas las demás columnas se conservaron porque alimentan alguno de los 4 KPIs.